# 📈 UppCar — Business Growth Forecasting (XGBoost)

**Objectif** : Prédire le nombre futur de réservations, clients et agences.
**Méthode** : Régression avec **XGBoost** avec validation Train/Test (80/20).

**Étapes** :
1. Uploader votre fichier `historical_data.csv`.
2. Entraînement avec détection d'Overfitting.
3. Télécharger le fichier `growth_forecast.json`.

In [ ]:
from google.colab import files
import io
import pandas as pd
import numpy as np
import json
from datetime import datetime, timedelta
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

print('Veuillez uploader le fichier historical_data.csv :')
uploaded = files.upload()
filename = list(uploaded.keys())[0]
df = pd.read_csv(io.BytesIO(uploaded[filename]))
print(f'✅ Fichier {filename} chargé avec {len(df)} lignes.')

LoadError: UndefVarError: `from` not defined in `Main`
Suggestion: check for spelling errors or missing imports.

In [ ]:
df['date'] = pd.to_datetime(df['date'])
df['day_index'] = np.arange(len(df))

targets = ['reservations', 'clients', 'agencies']
predictions = {}

for target in targets:
    print(f'--- Analyzing: {target} ---')
    X = df[['day_index']]
    y = df[target]
    
    # Split 80% train / 20% test
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    # Hyper-paramètres optimisés pour éviter l'overfitting sur petit dataset
    model = XGBRegressor(
        n_estimators=100, 
        learning_rate=0.05, 
        max_depth=3, 
        reg_lambda=1, # L2 regularization
        objective='reg:squarederror'
    )
    
    model.fit(X_train, y_train, eval_set=[(X_train, y_train), (X_test, y_test)], verbose=False)

    # Évaluation RMSE
    train_rmse = np.sqrt(mean_squared_error(y_train, model.predict(X_train)))
    test_rmse = np.sqrt(mean_squared_error(y_test, model.predict(X_test)))
    
    print(f'Train RMSE: {train_rmse:.4f}')
    print(f'Test RMSE: {test_rmse:.4f}')
    
    if train_rmse < test_rmse * 0.7:
        print('⚠️ WARNING: Potential Overfitting detected (Train RMSE much lower than Test)')
    else:
        print('✅ Model stability: Good')

    # Entraînement final sur TOUTES les données pour la prédiction future
    model.fit(X, y)
    
    # Prévoir les 30 prochains jours
    future_days = np.arange(len(df), len(df) + 30).reshape(-1, 1)
    future_preds = model.predict(pd.DataFrame(future_days, columns=['day_index']))
    
    predictions[target] = {
        'historical': y.tolist(),
        'forecast': [max(0, float(p)) for p in future_preds],
        'dates': [(df['date'].max() + timedelta(days=i+1)).strftime('%Y-%m-%d') for i in range(30)],
        'metrics': {'rmse': float(test_rmse)}
    }
    print(f'Forecast generated for {target}.\n')

with open('growth_forecast.json', 'w') as f:
    json.dump(predictions, f, indent=4)

print('🚀 Fichier growth_forecast.json créé avec succès !')

In [ ]:
files.download('growth_forecast.json')
print('✅ Téléchargement lancé.')